
# Q3 Long-time Drying Analysis

This notebook extends the validated Q2 solver to Question 3.

**Production numerical route**

\[
\boxed{
\text{cylindrical cell-centered FVM}
+\text{joint BDF}
+\text{local coupled Robin Picard}
+\text{long-term boundary extension}
+\text{terminal event}
}
\]

The formal drying event is

\[
g(t)=C_{\max}(t)-0.15=0,
\]

where \(C_{\max}\) is evaluated from the **full FVM state**, including the reconstructed axis and surface. It is not evaluated only on the 0.1 cm output grid.

The notebook separates three time concepts:

- solver internal adaptive step;
- `result3.xlsx` sampling interval: 60 s;
- drying-event localization accuracy, which can be much finer than 60 s.

> The first moderate-grid solve is for development/inspection. Final paper numbers should be regenerated with the production settings near the end of the notebook.


In [ ]:

from pathlib import Path
from dataclasses import replace
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
if not (PROJECT / "src").exists() and (PROJECT.parent / "src").exists():
    PROJECT = PROJECT.parent
if not (PROJECT / "src").exists():
    raise RuntimeError("Run this notebook from the Q3_drying_project root or notebooks/ directory.")

sys.path.insert(0, str(PROJECT / "src"))

from q2_solver import (
    density, heat_capacity, thermal_conductivity, moisture_diffusivity
)
from q3_solver import (
    Q3Parameters, Q3SolverConfig, LongTermEnvironment,
    solve_q3, find_drying_time,
    environment_summary, drying_event_summary,
    cmax_location_history, physical_diagnostics_q3,
    paper_table5, event_local_trace,
    long_time_balance_diagnostics,
    event_grid_convergence, event_time_sensitivity,
    boundary_strategy_sensitivity,
    find_drying_time_block_picard,
    write_result3,
)

ENV_PATH = PROJECT / "data" / "附件1.xlsx"
OUT_DIR = PROJECT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Environment file exists:", ENV_PATH.exists())


## 1. Inspect the measured boundary and the long-term extension

In [ ]:

env_main = LongTermEnvironment.from_excel(ENV_PATH, "last_hour_mean")
display(environment_summary(env_main))

env_last = LongTermEnvironment.from_excel(ENV_PATH, "last_value")
env_15h = LongTermEnvironment.from_excel(ENV_PATH, "last_1p5h_mean")

print("Measured data end:", env_main.data_end_s / 3600, "h")
print("Main plateau T:", env_main.stats.T_long_C, "degC")
print("Main plateau C datum:", env_main.stats.C_long)


In [ ]:

t_plot = np.linspace(0, 24*3600, 1201)

plt.figure(figsize=(8, 4.5))
plt.plot(env_main.base.t/3600, env_main.base.temperature, marker="o", markersize=2, label="Measured")
plt.plot(t_plot/3600, env_main.T_inf(t_plot), label="Main extension")
plt.axvline(env_main.data_end_s/3600, linestyle="--", label="Data end")
plt.xlabel("Time (h)")
plt.ylabel("Ambient temperature (degC)")
plt.title("Q3 ambient temperature and long-term extension")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(env_main.base.t/3600, env_main.base.moisture, marker="o", markersize=2, label="Measured")
plt.plot(t_plot/3600, env_main.C_inf(t_plot), label="Main extension")
plt.axvline(env_main.data_end_s/3600, linestyle="--", label="Data end")
plt.xlabel("Time (h)")
plt.ylabel("Ambient moisture datum (kg/kg)")
plt.title("Q3 ambient moisture datum and long-term extension")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



The main extrapolation intentionally follows the modeling handoff:

- \(0\le t\le4\) h: measured data with piecewise-linear interpolation;
- \(t>4\) h: constant mean of the measured 3-4 h platform.

Because the 4 h last measured value is not exactly equal to the 3-4 h mean, the main rule contains a small boundary jump at 4 h. The solver restarts at that point, so this is handled numerically rather than smoothed silently.


## 2. Material-property scales at the low-moisture stage

In [ ]:

p = Q3Parameters()
C_grid = np.linspace(0.08, 2.55, 500)

print("Initial D:", moisture_diffusivity(p.C0, p.T0_C))
print("D(C=2.55,T=50C):", moisture_diffusivity(2.55, 50.0))
print("D(C=0.15,T=50C):", moisture_diffusivity(0.15, 50.0))
print("D ratio 2.55 / 0.15 at 50C:",
      moisture_diffusivity(2.55, 50.0) / moisture_diffusivity(0.15, 50.0))

plt.figure(figsize=(7.5, 4.5))
plt.semilogy(C_grid, moisture_diffusivity(C_grid, 50.0))
plt.axvline(0.15, linestyle="--", label="Drying threshold")
plt.xlabel("Moisture content C (kg/kg)")
plt.ylabel("Diffusivity D (m^2/s)")
plt.title("Moisture diffusivity at 50 degC")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



The sharp decrease of \(D(C,T)\) as \(C\to0.15\) is why Q3 is not simply "run Q2 longer." The final stage is much slower and is also where the nonlinear surface Robin equation becomes numerically more demanding.


## 3. Development event-time preview

In [ ]:

# Faster preview settings for development only.
# Do NOT use these values as the final paper settings without the convergence cells below.
cfg_preview = Q3SolverConfig(
    rtol=2e-8,
    atol=2e-10,
    max_step_data=30.0,
    max_step_long=300.0,
    output_dt=60.0,
    long_segment_s=6*3600.0,
    event_max_step=5.0,
    max_time_s=8*24*3600.0,
    segment_at_data_knots=False,
    refine_event=True,
)

preview_event = find_drying_time(
    ENV_PATH, N=320, config=cfg_preview, boundary_strategy="last_hour_mean"
)
print(preview_event)
print("Development estimate, tdry =", preview_event.t_dry_h, "h")
print("Controlling radius =", 100*preview_event.rmax_m, "cm")
print("Event refinement shift =", preview_event.solver_stats.get("refinement_shift_s"), "s")
print("Surface Brent fallback calls =", preview_event.solver_stats.get("surface_brent_fallback_calls"))



A coarse-grid warning is important for Q3. Very coarse grids can make the nonlinear surface Robin algebraic equation lose the contraction needed by pure Picard before the center reaches \(0.15\). The formal convergence sequence should therefore start from a grid that actually lies in the asymptotic regime. In current smoke tests, \(N=320,640,1280\) are the useful sequence; \(N=160\) is too coarse for a reliable event-time study.


## 4. Full moderate-grid solution to the terminal event

In [ ]:

# This stores the required 60 s x 0.1 cm field for analysis.
sol = solve_q3(
    ENV_PATH,
    N=320,
    config=cfg_preview,
    boundary_strategy="last_hour_mean",
)

print("Output shape T:", sol.T.shape)
print("Output shape C:", sol.C.shape)
print("Last regular output time:", sol.t[-1], "s")
print("Exact drying time:", sol.t_dry_s, "s =", sol.t_dry_s/3600, "h")
print("Solver stats:", sol.solver_stats)
display(pd.DataFrame([drying_event_summary(sol)]))


## 5. Table 5

In [ ]:

table5 = paper_table5(sol)
display(table5.round(4))


## 6. Verify that the full-domain maximum controls the event

In [ ]:

hist = cmax_location_history(sol)
display(hist.tail(12))

print("Fraction of monitored full states whose maximum is at the axis:",
      np.mean(np.isclose(hist["rmax_cm"], 0.0)))

plt.figure(figsize=(8, 4.5))
plt.plot(hist["time_h"], hist["Cmax"], label="Full-state Cmax")
plt.axhline(sol.params.threshold, linestyle="--", label="Threshold")
plt.xlabel("Time (h)")
plt.ylabel("Maximum moisture content (kg/kg)")
plt.title("Full-domain maximum moisture versus threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(hist["time_h"], hist["rmax_cm"])
plt.xlabel("Time (h)")
plt.ylabel("Radius of maximum moisture (cm)")
plt.title("Location controlling the drying criterion")
plt.grid(True, alpha=0.3)
plt.show()


## 7. Long-time radial moisture profiles

In [ ]:

profile_hours = list(np.arange(6, np.floor(sol.t_dry_s/3600/6)*6 + 0.1, 6))
if len(profile_hours) > 8:
    # Keep the plot readable while preserving early/late stages.
    profile_hours = profile_hours[::2]
if sol.t_dry_s/3600 - profile_hours[-1] > 0.2:
    profile_hours.append(sol.t_dry_s/3600)

plt.figure(figsize=(8, 4.8))
for h in profile_hours:
    if abs(h*3600 - sol.t_dry_s) < 1.0:
        # Event snapshot: sample the exact full state.
        from q3_solver import sample_state_pair
        _, Cprof = sample_state_pair(
            sol.T_dry_state, sol.C_dry_state, sol.t_dry_s,
            sol.r, sol.grid, sol.env, sol.params, sol.config.rhs_config()
        )
    else:
        j = int(np.argmin(np.abs(sol.t - h*3600)))
        Cprof = sol.C[j]
    plt.plot(sol.r*100, Cprof, label=f"{h:.1f} h")

plt.axhline(sol.params.threshold, linestyle="--", label="Threshold")
plt.xlabel("Radius from center (cm)")
plt.ylabel("Moisture content (kg/kg)")
plt.title("Q3 long-time radial moisture profiles")
plt.legend(ncol=2)
plt.grid(True, alpha=0.3)
plt.show()


## 8. Center, mid-radius and surface histories

In [ ]:

idx_center = np.where(np.isclose(sol.r, 0.0))[0][0]
idx_mid = np.where(np.isclose(sol.r, 0.01))[0][0]
idx_surface = np.where(np.isclose(sol.r, 0.02))[0][0]

plt.figure(figsize=(8, 4.8))
plt.plot(sol.t/3600, sol.C[:, idx_center], label="Center")
plt.plot(sol.t/3600, sol.C[:, idx_mid], label="1 cm")
plt.plot(sol.t/3600, sol.C[:, idx_surface], label="Surface")
plt.axhline(sol.params.threshold, linestyle="--", label="Threshold")
plt.xlabel("Time (h)")
plt.ylabel("Moisture content (kg/kg)")
plt.title("Q3 long-time moisture histories")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 9. State-dependent diffusivity and late-stage slowdown

In [ ]:

D_center = moisture_diffusivity(sol.C[:, idx_center], sol.T[:, idx_center])
D_mid = moisture_diffusivity(sol.C[:, idx_mid], sol.T[:, idx_mid])
D_surface = moisture_diffusivity(sol.C[:, idx_surface], sol.T[:, idx_surface])

plt.figure(figsize=(8, 4.8))
plt.semilogy(sol.t/3600, D_center, label="Center")
plt.semilogy(sol.t/3600, D_mid, label="1 cm")
plt.semilogy(sol.t/3600, D_surface, label="Surface")
plt.xlabel("Time (h)")
plt.ylabel("Diffusivity D (m^2/s)")
plt.title("Long-time state-dependent moisture diffusivity")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 10. Local event crossing: accuracy independent of the 60 s output grid

In [ ]:

trace = event_local_trace(sol, before_s=600.0, after_s=300.0, dt=5.0)
display(trace.iloc[(trace["relative_to_event_s"].abs()).argsort()[:10]].sort_values("time_s"))

plt.figure(figsize=(8, 4.5))
plt.plot(trace["relative_to_event_s"], trace["g"])
plt.axhline(0.0, linestyle="--")
plt.axvline(0.0, linestyle="--")
plt.xlabel("Time relative to drying event (s)")
plt.ylabel("g(t) = Cmax - 0.15")
plt.title("Local terminal-event crossing")
plt.grid(True, alpha=0.3)
plt.show()


## 11. Physical and semi-discrete balance diagnostics

In [ ]:

print("Physical diagnostics:")
print(physical_diagnostics_q3(sol))

bal = long_time_balance_diagnostics(sol)
print("Heat instantaneous max relative residual:",
      bal["heat_instantaneous_max_relative"])
print("Moisture instantaneous max relative residual:",
      bal["moisture_instantaneous_max_relative"])
print("Moisture integral final relative residual:",
      bal["moisture_integral_final_relative"])
print("Moisture integral global-normalized max residual:",
      bal["moisture_integral_global_normalized_max"])

plt.figure(figsize=(8, 4.2))
plt.semilogy(
    bal["state_time_s"]/3600,
    np.abs(bal["moisture_integral_residual"]) + 1e-30
)
plt.xlabel("Time (h)")
plt.ylabel("Integral-balance residual")
plt.title("Q3 moisture integral-balance residual")
plt.grid(True, alpha=0.3)
plt.show()



For heat, use the **instantaneous semi-discrete balance** rather than the Q1 cumulative-energy formula, because \(\rho(C)c_p(C)\) varies with moisture.


## 12. Drying-time grid convergence — main Q3 convergence metric

In [ ]:

# Production-like event configuration. This cell can be computationally expensive.
cfg_event = Q3SolverConfig(
    rtol=1e-9,
    atol=1e-11,
    max_step_data=5.0,
    max_step_long=60.0,
    event_max_step=5.0,
    output_dt=60.0,
    long_segment_s=3600.0,
    max_time_s=8*24*3600.0,
    segment_at_data_knots=True,
    refine_event=True,
)

# Recommended formal sequence: 320 -> 640 -> 1280.
# If runtime is limited, first run (320, 640), then add 1280.
RUN_GRID_CONVERGENCE = False

if RUN_GRID_CONVERGENCE:
    conv_df, conv_results = event_grid_convergence(
        ENV_PATH,
        N_values=(320, 640, 1280),
        config=cfg_event,
        boundary_strategy="last_hour_mean",
    )
    display(conv_df)

    if len(conv_df) >= 2:
        d1 = conv_df.iloc[0]["abs_difference_s"]
        d2 = conv_df.iloc[1]["abs_difference_s"]
        print("Difference ratio coarse/fine:", d1/d2)
        print("Observed order estimate log2(ratio):", np.log2(d1/d2))
else:
    print("Set RUN_GRID_CONVERGENCE=True for the formal 320/640/1280 event-time convergence run.")



A development smoke run with looser controls gave approximately:

- \(N=320:\ 57.6681\) h;
- \(N=640:\ 57.5164\) h;
- \(N=1280:\ 57.4853\) h.

These are **not final paper values**. Their purpose is to show that the implementation reaches the event and that the event time is already converging strongly with grid refinement. Recompute with `cfg_event` before reporting final numbers.


## 13. Event-time tolerance / max-step sensitivity

In [ ]:

RUN_TIME_SENSITIVITY = False

if RUN_TIME_SENSITIVITY:
    time_df = event_time_sensitivity(
        ENV_PATH,
        N=320,
        baseline=cfg_event,
        boundary_strategy="last_hour_mean",
    )
    display(time_df)
else:
    print("Set RUN_TIME_SENSITIVITY=True to compare baseline and tighter BDF/event controls.")


## 14. Long-term boundary extrapolation sensitivity

In [ ]:

RUN_BOUNDARY_SENSITIVITY = False

if RUN_BOUNDARY_SENSITIVITY:
    bc_df = boundary_strategy_sensitivity(
        ENV_PATH,
        N=320,
        config=cfg_event,
    )
    display(bc_df)

    tdry = bc_df["tdry_h"].to_numpy()
    print("Drying-time spread (h):", tdry.max() - tdry.min())
    print("Relative spread:",
          (tdry.max()-tdry.min()) /
          bc_df.loc[bc_df["strategy"]=="last_hour_mean", "tdry_h"].iloc[0])
else:
    print("Set RUN_BOUNDARY_SENSITIVITY=True for the three required long-boundary scenarios.")



The boundary sensitivity is a **model-extrapolation uncertainty**, not a numerical discretization error. Keep it separate from grid and time convergence in the paper.


## 15. Surface nonlinear solver diagnostics

In [ ]:

print("Surface Picard success calls:",
      sol.solver_stats.get("surface_picard_success_calls"))
print("Surface continuation-root fallback calls:",
      sol.solver_stats.get("surface_brent_fallback_calls"))
print("Surface mean iterations:",
      sol.solver_stats.get("surface_mean_iterations"))
print("Surface max iteration counter:",
      sol.solver_stats.get("surface_max_iterations"))

print(
    "\nInterpretation: the fallback solves the same local Robin fixed-point equation "
    "and always selects the largest root connected to the near-surface interior state. "
    "On sufficiently fine grids (e.g. current N=640/1280 smoke tests), the fallback "
    "is normally not needed at all."
)


## 16. Optional backward-Euler + block-Picard long-time stress test

In [ ]:

# This is deliberately an auxiliary stress test, not the production solver.
# In the low-moisture long-time regime, coarse fixed-step block Picard may cease
# to converge. That failure itself is useful evidence for keeping joint BDF as
# the formal Q3 method.

RUN_BLOCK_PICARD_STRESS = False

if RUN_BLOCK_PICARD_STRESS:
    try:
        bp = find_drying_time_block_picard(
            ENV_PATH,
            N=80,
            dt=600.0,
            max_time_s=8*24*3600.0,
            boundary_strategy="last_hour_mean",
        )
        print(bp)
    except RuntimeError as exc:
        print("Block-Picard long-time stress test did not converge:")
        print(exc)
        print("This does not invalidate the joint-BDF production solution.")
else:
    print("Set RUN_BLOCK_PICARD_STRESS=True only for the auxiliary long-time stress test.")


## 17. Final N=1280 production solve

In [ ]:

# This is the final production cell and can take substantially longer.
RUN_PRODUCTION = False

if RUN_PRODUCTION:
    sol_prod = solve_q3(
        ENV_PATH,
        N=1280,
        config=cfg_event,
        boundary_strategy="last_hour_mean",
    )
    display(pd.DataFrame([drying_event_summary(sol_prod)]))
    display(paper_table5(sol_prod).round(4))
    print(physical_diagnostics_q3(sol_prod))
    print(sol_prod.solver_stats)
else:
    print("Set RUN_PRODUCTION=True after the convergence/sensitivity checks are satisfactory.")


## 18. Write result3.xlsx

In [ ]:

TEMPLATE_PATH = PROJECT / "data" / "result3_template.xlsx"
if not TEMPLATE_PATH.exists():
    TEMPLATE_PATH = None

if "sol_prod" in globals():
    OUT_PATH = OUT_DIR / "result3.xlsx"
    write_result3(sol_prod, OUT_PATH, template_path=TEMPLATE_PATH, decimals=4)
    print("Saved:", OUT_PATH)
    print("Exact drying time (report separately):",
          sol_prod.t_dry_s, "s =", sol_prod.t_dry_s/3600, "h")
else:
    print("Run the final production cell first. The writer intentionally does not append "
          "the non-60-s event time as an extra result3 row.")



## 19. Final interpretation checklist

Before Q3 is considered finished:

- verify \(C>0\) and \(D>0\) throughout;
- verify \(C_{\max}(t)\) decreases smoothly near the terminal event;
- verify the controlling location \(r_{\max}(t)\), rather than assuming it is the center;
- report the exact event time independently of the 60 s Excel sampling;
- use \(t_{\rm dry}\), not a fixed-time field error, as the primary Q3 grid-convergence metric;
- keep long-boundary extrapolation sensitivity separate from numerical error;
- use the 320/640/1280 convergence sequence;
- write `result3.xlsx` only after the final N=1280 solution is accepted.

### Execution route

`Q2 validated FVM core`
→ `measured boundary 0-4 h`
→ `platform extension after 4 h`
→ `joint BDF`
→ `local coupled Robin solve`
→ `full-state Cmax event`
→ `local event refinement`
→ `tdry grid convergence`
→ `time/event sensitivity`
→ `boundary extrapolation sensitivity`
→ `Table 5`
→ `result3.xlsx`.
